# Can March Signals Prioritize Content for April Review?

**Ashritha Chary** · FlyRank ML Internship capstone · 29 September 2026

## Abstract

This study asks whether March measurements can rank content items at higher risk of an April impressions drop for limited editorial review. It queries the FlyRank internship warehouse's daily performance fact table and content dimension, yielding 143,206 eligible March-April content/client pairs across 45 clients from a release containing about 79 million daily records. Regularized logistic regression and histogram gradient boosting were compared with a fixed Week-4 rule using five client-held-out folds, while random-row splits were retained only as a weaker diagnostic. Across client-held-out folds, logistic regression achieved mean precision@50 of 0.688 versus 0.392 for the rule and a test base rate of 0.517; because model selection used those same folds, this result is exploratory rather than an untouched test. The result supports further evaluation of human-review prioritization, but it does not establish refresh uplift, predict search rankings, or guarantee performance on future months.

[Open the deployed research paper](https://ashritha-boop.github.io/fly_machine/) · [Repository](https://github.com/Ashritha-boop/fly_machine)

## 1. Introduction / Problem statement

A content editor has limited time to review pages whose search visibility may be weakening. This paper asks: **using measurements available by 31 March 2026, can a model rank eligible content items that will show a large observed impressions drop in April more effectively than a transparent rule?** The intended decision is which items to inspect first, not which pages to edit automatically.

The unit is one pseudonymized content item within a client. A false positive spends review time on an item that does not cross the proxy threshold; a false negative leaves a declining item lower in the review order. Precision@50 therefore measures the share of the first 50 ranked items that meet the proxy label, while the test base rate provides context.

In [1]:
RESEARCH_QUESTION = (
    "Do March 2026 measurements rank content items with an April impressions-drop "
    "proxy more effectively than the fixed Week-4 review rule?"
)
PREDICTION_CUTOFF = "2026-03-31"
LABEL_WINDOW = ("2026-04-01", "2026-04-30")
print(RESEARCH_QUESTION)
print(f"Feature cutoff: {PREDICTION_CUTOFF}; label window: {LABEL_WINDOW[0]} to {LABEL_WINDOW[1]}")

Do March 2026 measurements rank content items with an April impressions-drop proxy more effectively than the fixed Week-4 review rule?
Feature cutoff: 2026-03-31; label window: 2026-04-01 to 2026-04-30


## 2. Data

The analysis uses the gated **FlyRank/internship-warehouse** Parquet release, described by the project as roughly 79 million daily search-performance rows. It queries `fact_content_daily_performance` and joins `dim_content` on pseudonymous content and client hashes. The evaluated frame is not 79 million observations: it contains **143,206 eligible content-client pairs across 45 clients** after aggregation.

March 1–31, 2026 supplies the feature window; April 1–30, 2026 supplies the outcome window only. The query keeps pairs observed in both months and requires at least 10 March impressions, then excludes rows missing required March measures or content age. GA4 sessions are used only when `ga4_data_available IS TRUE`; availability itself is retained as a feature. The label is 1 when April impressions are below 80% of March impressions.

This is a paired-month subset of the internship release, not a claim that the model trained over all 79 million daily rows. Hashed IDs are used only to join rows and form client-grouped folds; no client names, domains, page URLs, titles, query text, raw rows, or identifiers are included in public outputs. Full-release access requires accepting the data-use terms; this notebook reads from the authorized warehouse and does not download or commit it.

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

if importlib.util.find_spec("duckdb") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb"])
import duckdb
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "work").is_dir():
    ROOT = Path("/content/fly_machine")
    if not (ROOT / "work").is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/Ashritha-boop/fly_machine.git", str(ROOT)],
            check=True,
        )

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
if not HF_TOKEN:
    raise RuntimeError(
        "Add the read-only HF_TOKEN to Colab Secrets after accepting the dataset terms. "
        "For local runs, set HF_TOKEN in the environment. No synthetic fallback is used."
    )

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
safe_token = HF_TOKEN.replace("'", "''")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{safe_token}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

paired_month_sql = f"""
WITH march AS (
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS gsc_impressions_march,
        SUM(gsc_clicks) AS gsc_clicks_march,
        AVG(gsc_avg_position) AS gsc_avg_position_march,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END)
            AS ga4_sessions_march,
        CASE WHEN BOOL_OR(ga4_data_available IS TRUE) THEN 1 ELSE 0 END
            AS ga4_available_march
    FROM {FACT}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
    GROUP BY content_hash_id, client_hash_id
),
april AS (
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS gsc_impressions_april
    FROM {FACT}
    WHERE report_date BETWEEN '2026-04-01' AND '2026-04-30'
    GROUP BY content_hash_id, client_hash_id
)
SELECT
    m.content_hash_id,
    m.client_hash_id,
    m.gsc_impressions_march,
    m.gsc_clicks_march,
    m.gsc_avg_position_march,
    m.ga4_sessions_march,
    m.ga4_available_march,
    DATEDIFF('day', c.content_created_date, DATE '2026-03-31') AS content_age_days,
    a.gsc_impressions_april,
    CASE WHEN a.gsc_impressions_april < 0.80 * m.gsc_impressions_march
         THEN 1 ELSE 0 END AS is_declining_label
FROM march m
JOIN april a
  ON m.content_hash_id = a.content_hash_id
 AND m.client_hash_id = a.client_hash_id
JOIN {CONTENT} c
  ON m.content_hash_id = c.content_hash_id
 AND m.client_hash_id = c.client_hash_id
WHERE m.gsc_impressions_march >= 10
"""
df_data = con.sql(paired_month_sql).df()
df_data["ga4_sessions_march"] = df_data["ga4_sessions_march"].where(
    df_data["ga4_available_march"].eq(1)
)
df_data = df_data.dropna(
    subset=[
        "gsc_impressions_march",
        "gsc_clicks_march",
        "gsc_avg_position_march",
        "content_age_days",
        "is_declining_label",
    ]
).reset_index(drop=True)
assert not df_data.duplicated(["client_hash_id", "content_hash_id"]).any()
assert df_data["is_declining_label"].isin([0, 1]).all()
print(
    f"Eligible paired-month rows: {len(df_data):,}; "
    f"clients: {df_data['client_hash_id'].nunique():,}; "
    f"proxy base rate: {df_data['is_declining_label'].mean():.3f}."
)
print("No identifiers or row-level examples are displayed or exported.")

## 3. Methodology

**Assumption and label.** The April proxy is a measurable decline screen, not a definition of harmful content or a future business outcome: 1 when April impressions are <80% of March impressions, otherwise 0. Both months must be observed, and the March impression floor of 10 defines the population. Conclusions are conditional on these eligibility rules.

**Features.** The seven March-only inputs are impressions, clicks, average position, GA4 sessions, content age as of March 31, the GA4-availability flag, and a position-availability flag. Missing GA4 sessions are imputed with the training-fold median in each model pipeline. Client/content hashes, April impressions, the label, trend-derived fields, and product decision scores are forbidden as features.

**Models and baseline.** The candidates are median-imputed, standardized Logistic Regression and median-imputed HistGradientBoosting. The fixed Week-4 rule scores an item only when it is at least 180 days old, has at least 500 March impressions, and average position is at least 4; its score is impressions divided by position plus one. Precision@50 and average precision are compared on identical test rows.

**Validation and leakage checks.** Five-fold GroupKFold keeps each client wholly in one fold (45 clients total); the same candidates and rule are also measured under five 80/20 random-row splits as a deliberately weaker diagnostic. Seed: 42. The winning model is selected by mean grouped-fold precision@50 on those same folds, so the result is exploratory and can be optimistic. Checks assert that client/content IDs and April/label/product fields are absent from `X`, March features precede April labels, and group IDs do not overlap across train/test. This is not a sealed holdout or a multi-month time-forward test.

In [ ]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.model_selection import GroupKFold, ShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

FEATURES = [
    "gsc_impressions_march",
    "gsc_clicks_march",
    "gsc_avg_position_march",
    "ga4_sessions_march",
    "content_age_days",
    "ga4_available_march",
    "position_available_march",
]
df_data["position_available_march"] = df_data["gsc_avg_position_march"].gt(0).astype(int)
FORBIDDEN_FEATURES = {
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions_april",
    "is_declining_label",
    "trend_direction",
    "trend_pct",
    "health_score",
    "priority_score",
    "action_type",
}
assert not (set(FEATURES) & FORBIDDEN_FEATURES)
X = df_data[FEATURES].copy()
y = df_data["is_declining_label"].astype(int).copy()
groups = df_data["client_hash_id"].copy()

SEED = 42
model_templates = {
    "Logistic Regression": make_pipeline(
        SimpleImputer(strategy="median", keep_empty_features=True),
        StandardScaler(),
        LogisticRegression(max_iter=2000, random_state=SEED),
    ),
    "HistGradientBoosting": make_pipeline(
        SimpleImputer(strategy="median", keep_empty_features=True),
        HistGradientBoostingClassifier(
            max_iter=120,
            max_leaf_nodes=15,
            min_samples_leaf=30,
            l2_regularization=1.0,
            random_state=SEED,
        ),
    ),
}

position = df_data["gsc_avg_position_march"].fillna(0)
rule_scores = (
    df_data["content_age_days"].ge(180).astype(int)
    * df_data["gsc_impressions_march"].ge(500).astype(int)
    * position.ge(4.0).astype(int)
    * df_data["gsc_impressions_march"]
    / (position + 1.0)
).to_numpy()


def precision_at_k(labels, scores, k=50):
    labels = np.asarray(labels)
    scores = np.asarray(scores)
    order = np.argsort(-scores, kind="mergesort")[: min(k, len(labels))]
    return float(labels[order].mean())


split_designs = {
    "Random row (diagnostic)": list(
        ShuffleSplit(n_splits=5, test_size=0.20, random_state=SEED).split(X, y)
    ),
    "Client-held-out (primary)": list(
        GroupKFold(n_splits=5).split(X, y, groups=groups)
    ),
}
metric_rows = []
grouped_oof_scores = {name: np.full(len(df_data), np.nan) for name in model_templates}
for design_name, splits in split_designs.items():
    for fold_number, (train_idx, test_idx) in enumerate(splits, start=1):
        if design_name == "Client-held-out (primary)":
            assert set(groups.iloc[train_idx]).isdisjoint(set(groups.iloc[test_idx]))
        y_test = y.iloc[test_idx]
        test_methods = {"Week-4 rule": rule_scores[test_idx]}
        for name, template in model_templates.items():
            fitted = clone(template).fit(X.iloc[train_idx], y.iloc[train_idx])
            scores = fitted.predict_proba(X.iloc[test_idx])[:, 1]
            test_methods[name] = scores
            if design_name == "Client-held-out (primary)":
                grouped_oof_scores[name][test_idx] = scores
        for method, scores in test_methods.items():
            metric_rows.append(
                {
                    "split": design_name,
                    "fold": fold_number,
                    "method": method,
                    "test_rows": len(test_idx),
                    "base_rate": float(y_test.mean()),
                    "precision_at_50": precision_at_k(y_test, scores, 50),
                    "average_precision": average_precision_score(y_test, scores),
                }
            )

assert all(np.isfinite(scores).all() for scores in grouped_oof_scores.values())
metrics = pd.DataFrame(metric_rows)
summary = metrics.groupby(["split", "method"], sort=False).agg(
    folds=("fold", "nunique"),
    mean_test_base_rate=("base_rate", "mean"),
    mean_precision_at_50=("precision_at_50", "mean"),
    sd_precision_at_50=("precision_at_50", "std"),
    mean_average_precision=("average_precision", "mean"),
).reset_index()
grouped_summary = summary[summary["split"] == "Client-held-out (primary)"].set_index("method")
best_model_name = grouped_summary.loc[list(model_templates), "mean_precision_at_50"].idxmax()

assert "is_declining_label" not in X.columns
assert "gsc_impressions_april" not in X.columns
assert not {"client_hash_id", "content_hash_id"} & set(X.columns)
print(
    f"Frame: {len(df_data):,} rows / {groups.nunique():,} clients; "
    f"sklearn {sklearn.__version__}; seed {SEED}."
)
print("PASS: no IDs, April outcome, target-derived, or product-output columns in model features.")
print(f"Selected by grouped-fold mean Precision@50: {best_model_name} (selection is exploratory).")

## 4. Results (vs baseline)

On the same five client-held-out folds and the same April proxy, Logistic Regression measured mean precision@50 **0.688 ± 0.157**, versus **0.392 ± 0.214** for the unchanged Week-4 rule; the mean test base rate was **0.517**. The model’s measured gap was +0.296 over the rule and +0.171 over the base rate. Mean average precision was 0.560 for both Logistic Regression and HistGradientBoosting, compared with 0.512 for the rule. HistGradientBoosting had lower grouped-fold precision@50 (0.524 ± 0.248).

The random-row diagnostic illustrates why client grouping matters: HistGradientBoosting measured 0.964 precision@50 with random rows, but 0.524 with client-held-out folds. Treat the grouped result as the relevant, still exploratory estimate; fold standard deviations describe variability and are not confidence intervals.

The chart and table are regenerated below from all five folds. Precision@50 is a ranking metric for this proxy, not expected recovery or calibrated probability.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

WORK_OUTPUTS = ROOT / "work" / "outputs"
WORK_FIGURES = ROOT / "work" / "figures"
WORK_OUTPUTS.mkdir(parents=True, exist_ok=True)
WORK_FIGURES.mkdir(parents=True, exist_ok=True)

print("Fold-mean metrics; standard deviations are descriptive across folds:")
display(summary.round(3))

plot_methods = ["Week-4 rule", "Logistic Regression", "HistGradientBoosting"]
plot_colors = {"Week-4 rule": "#6B7478", "Logistic Regression": "#147D73", "HistGradientBoosting": "#E0783E"}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), sharey=True, layout="constrained")
for ax, split_name, title in [
    (axes[0], "Random row (diagnostic)", "Random rows · diagnostic"),
    (axes[1], "Client-held-out (primary)", "Clients held out · primary"),
]:
    values = summary[summary["split"] == split_name].set_index("method")
    means = [values.loc[name, "mean_precision_at_50"] for name in plot_methods]
    spreads = [values.loc[name, "sd_precision_at_50"] for name in plot_methods]
    bars = ax.bar(
        range(len(plot_methods)), means, yerr=spreads, capsize=4,
        color=[plot_colors[name] for name in plot_methods], width=0.64,
        error_kw={"elinewidth": 1.4, "ecolor": "#344447"},
    )
    base_rate = values["mean_test_base_rate"].mean()
    ax.axhline(base_rate, color="#C34D35", linestyle=(0, (4, 3)), linewidth=1.5)
    ax.text(2.48, base_rate + 0.025, f"Base rate {base_rate:.3f}", ha="right", color="#8A3425", fontsize=9)
    ax.set_title(title, loc="left", fontweight="bold")
    ax.set_xticks(range(len(plot_methods)), ["Rule", "Logistic\nregression", "Gradient\nboosting"])
    ax.set_ylim(0, 1.12)
    ax.grid(axis="y", alpha=0.18)
    ax.set_axisbelow(True)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.bar_label(bars, labels=[f"{value:.3f}" for value in means], padding=5, fontsize=9)
axes[0].set_ylabel("Mean precision@50 across five folds")
fig.suptitle("Client grouping changes the ranking picture", x=0.02, ha="left", fontsize=15, fontweight="bold")
chart_path = WORK_FIGURES / "capstone_precision_at_50.png"
fig.savefig(chart_path, dpi=180, facecolor="white")
plt.show()
plt.close(fig)
print(f"Aggregate-only chart saved to {chart_path.relative_to(ROOT)}")

## 5. Limitations & honest framing

- The label is a one-month, thresholded impressions proxy, not a human judgment of page quality, an observed refresh response, or a future search outcome. A page crossing the 20% threshold does not imply that editing it would help.
- This is one March-to-April transition. The frame requires paired-month coverage and at least 10 March impressions; results do not automatically apply to unavailable, low-impression, or newly observed pages.
- Logistic Regression was selected on the same five client-held-out folds used to report its score. Grouped folds reduce client overlap but do not provide an untouched test or establish month-ahead transfer. Fold spread is descriptive, not a confidence interval.
- Probability scores were not calibrated. The illustrative 0.5 cutoff is arbitrary; classification error counts are not the evaluation target. Errors also varied by age band in this observed sample.
- No refresh interventions, recovered clicks, editorial effort, revenue, or user outcomes were measured. No causal effect, ROI, search-ranking prediction, or guaranteed uplift is supported.
- The source release is gated and mutable; the analysis records its repository path and date windows, but not a content-addressed release revision. Reruns may differ if the source or library versions change.

The audit used hashed IDs only for joins and grouped validation, checked that April outcomes and direct label/product fields were absent from model features, and emitted only aggregate diagnostics. A separate click-decline analysis in the validation notebook uses a different label and population; its metrics are not combined with this impressions study.

In [ ]:
selected_oof = grouped_oof_scores[best_model_name]
actual = y.to_numpy()
predicted = (selected_oof >= 0.5).astype(int)
error_frame = pd.DataFrame(
    {
        "content_age_days": df_data["content_age_days"],
        "any_error": predicted != actual,
        "false_positive": (predicted == 1) & (actual == 0),
        "false_negative": (predicted == 0) & (actual == 1),
    }
)
error_frame["age_band"] = pd.cut(
    error_frame["content_age_days"],
    bins=[-np.inf, 180, 365, np.inf],
    labels=["under 180 days", "180-365 days", "over 365 days"],
)
print(
    "Aggregate out-of-fold errors at the illustrative 0.5 threshold "
    "(scores are not calibrated):"
)
print(
    f"False positives: {int(error_frame['false_positive'].sum()):,}; "
    f"false negatives: {int(error_frame['false_negative'].sum()):,}."
)
display(
    error_frame.groupby("age_band", observed=False)
    .agg(rows=("any_error", "size"), error_rate=("any_error", "mean"))
    .round(3)
)
print("These are descriptive diagnostics only; no identifiers or per-row examples are shown.")

## 6. Ranked recommendations

1. **Keep the output as a review queue, not an editing command.** A grouped-fold mean precision@50 of 0.688 is encouraging for this proxy, but it was selected on the same folds; treat the model as a pilot candidate, not a production winner.
2. **Recheck eligibility and current evidence before reviewing a page.** Confirm that the March/April measurements are complete and current, inspect seasonal context and recent edits, and use the model only where the minimum-impression and paired-window assumptions hold.
3. **Put a human between every score and every change.** Editors should check search intent, current page quality, factual accuracy, compliance, and the reason for a proposed edit. Log accept, defer, or reject with a brief reason so the next evaluation can test real workflow value.
4. **Compare the ranking with the transparent rule and the base rate.** Keep all three visible in a capacity-limited pilot; do not use random-row validation as evidence of transfer to new clients.
5. **Require a later time-forward evaluation before wider use.** Repeat the study over several future months, preserve an untouched month for final testing, check calibration and client-level variation, and measure review cost and downstream outcomes before making any uplift or ROI claim.

No page should be auto-published, rewritten, deleted, redirected, canonicalized, or assigned from this score.

In [ ]:
recommendations = pd.DataFrame(
    [
        {
            "rank": 1,
            "recommendation": "Pilot as a human-reviewed ranking aid",
            "evidence": "Grouped-fold Precision@50 was 0.688; model selection used these same folds.",
            "guardrail": "Not a production winner or an editing command.",
        },
        {
            "rank": 2,
            "recommendation": "Verify paired-window eligibility and current measurements",
            "evidence": "The evaluated frame requires both months and at least 10 March impressions.",
            "guardrail": "Results do not cover unavailable or below-floor items.",
        },
        {
            "rank": 3,
            "recommendation": "Require editorial review and log decisions",
            "evidence": "No intervention, edit cost, or downstream outcome was measured.",
            "guardrail": "Never auto-publish, rewrite, delete, redirect, or assign work.",
        },
        {
            "rank": 4,
            "recommendation": "Retain the rule and base rate as comparators",
            "evidence": "Rule Precision@50 was 0.392; grouped test base rate was 0.517.",
            "guardrail": "Do not use random-row scores to claim new-client transfer.",
        },
        {
            "rank": 5,
            "recommendation": "Validate on later untouched months before scaling",
            "evidence": "This is one March-to-April transition with same-fold model selection.",
            "guardrail": "Measure calibration, reviewer effort, and outcomes before ROI claims.",
        },
    ]
)
display(recommendations)
assert recommendations["rank"].tolist() == [1, 2, 3, 4, 5]

## 7. Artifacts the paper embeds

The paper's paired-fold precision chart is rebuilt from this notebook and a public-safe SVG copy is committed with the report. The aggregate metrics receipt records the frame, target, split design, base rate, fold means/spread, and baseline comparison; no page- or client-level records are exported.

- [Deployed paper](https://ashritha-boop.github.io/fly_machine/)
- [Capstone notebook on GitHub](https://github.com/Ashritha-boop/fly_machine/blob/main/work/notebooks/capstone.ipynb)
- [Validation audit notebook](https://github.com/Ashritha-boop/fly_machine/blob/main/work/notebooks/w06_validation_audit.ipynb)
- [Repository and rerun instructions](https://github.com/Ashritha-boop/fly_machine)
- Generated files: `work/figures/capstone_precision_at_50.png` (created on rerun) and `work/outputs/capstone_metrics.json`
- [Committed aggregate SVG figure](../figures/capstone_precision_at_50.svg)

### Reproducibility

Open the capstone notebook in Colab, accept the FlyRank warehouse data-use terms, and add a read-only Hugging Face token as a Colab Secret named `HF_TOKEN`. Run all cells. The notebook queries only the March/April aggregates needed for this study, fixes the random seed at 42, prints aggregate-only results, and saves the chart and metrics receipt under `work/`. Never put the token in a notebook cell or commit it. The warehouse access is gated; a reader without approval can inspect the committed audit outputs and code but cannot fetch the data.

In [ ]:
import json


def records_for_split(split_name):
    selected = summary[summary["split"] == split_name]
    return {
        row["method"]: {
            "folds": int(row["folds"]),
            "mean_test_base_rate": float(row["mean_test_base_rate"]),
            "mean_precision_at_50": float(row["mean_precision_at_50"]),
            "sd_precision_at_50": float(row["sd_precision_at_50"]),
            "mean_average_precision": float(row["mean_average_precision"]),
        }
        for row in selected.to_dict(orient="records")
    }


receipt = {
    "study": "March 2026 features; April 2026 impressions-drop proxy",
    "source": {
        "dataset": "FlyRank/internship-warehouse",
        "fact_table": "fact_content_daily_performance",
        "dimension_table": "dim_content",
        "warehouse_release_scale": "approximately 79 million daily rows; not all rows are model observations",
        "analysis_rows": int(len(df_data)),
        "client_count": int(groups.nunique()),
        "feature_window": ["2026-03-01", "2026-03-31"],
        "label_window": ["2026-04-01", "2026-04-30"],
        "eligibility": "paired months, March impressions >= 10, required March fields and content age present",
        "label": "April impressions < 80% of March impressions",
        "proxy_base_rate": float(y.mean()),
    },
    "validation": {
        "primary_split": "5-fold GroupKFold by client; 45 clients",
        "diagnostic_split": "5 x 80/20 ShuffleSplit by row; weaker diagnostic only",
        "seed": SEED,
        "selected_model": best_model_name,
        "selection_metric": "mean grouped-fold precision_at_50; selected and evaluated on same folds",
        "primary_grouped_fold_means": records_for_split("Client-held-out (primary)"),
        "random_row_diagnostic_means": records_for_split("Random row (diagnostic)"),
        "fold_metrics": [
            {
                "split": row["split"],
                "fold": int(row["fold"]),
                "method": row["method"],
                "test_rows": int(row["test_rows"]),
                "base_rate": float(row["base_rate"]),
                "precision_at_50": float(row["precision_at_50"]),
                "average_precision": float(row["average_precision"]),
            }
            for row in metrics.to_dict(orient="records")
        ],
        "interpretation": "exploratory ranking on a same-fold-selected monthly proxy; no causal or future-month claim",
    },
    "leakage_checks": {
        "client_and_content_hashes_excluded_from_features": True,
        "april_outcome_and_proxy_label_excluded_from_features": True,
        "trend_and_product_decision_fields_excluded": True,
        "client_groups_disjoint_in_primary_folds": True,
    },
    "environment": {"scikit_learn": sklearn.__version__, "random_seed": SEED},
    "exports": {
        "figure": "work/figures/capstone_precision_at_50.png",
        "metrics": "work/outputs/capstone_metrics.json",
        "row_level_data_exported": False,
    },
}
receipt_path = WORK_OUTPUTS / "capstone_metrics.json"
receipt_path.write_text(json.dumps(receipt, indent=2, allow_nan=False) + "\n", encoding="utf-8")
assert chart_path.exists() and receipt_path.exists()
print(f"Aggregate chart: {chart_path.relative_to(ROOT)}")
print(f"Aggregate metrics receipt: {receipt_path.relative_to(ROOT)}")
print("Public-safe export check: row-level data and IDs were not written.")

## Demo & share (ML-12)

### Five-minute demo outline

- **0:00–1:00:** State the review-capacity question and the one-month impressions-drop proxy.
- **1:00–2:00:** Show the two-month warehouse frame, eligibility rule, and March-only feature boundary.
- **2:00–3:00:** Compare the Week-4 rule, Logistic Regression, and base rate on client-held-out folds; contrast with the random-row diagnostic.
- **3:00–4:00:** Walk through the leakage assertions and limitations: same-fold selection, one transition, and no measured refresh outcomes.
- **4:00–5:00:** Present the ranked human-review playbook and the next-month untouched validation needed before deployment.

### Social-post cut

I evaluated a March-to-April content-review ranking on 143,206 eligible content/client pairs from the FlyRank internship warehouse. Across five client-held-out folds, Logistic Regression measured mean precision@50 of 0.688 vs 0.392 for a fixed rule (base rate 0.517). This is an exploratory result on one monthly proxy, not evidence of refresh uplift; code and limitations are in the paper: [research paper](../../docs/index.html).

### Employer-facing summary

I built a reproducible, client-held-out evaluation to test whether March search and content signals could prioritize April impressions-drop reviews. On 143,206 eligible paired-month items across 45 clients, Logistic Regression measured mean precision@50 of 0.688 versus 0.392 for the fixed rule, against a 0.517 test base rate. The result is an exploratory decision-support signal; model selection shared the evaluation folds, and no refresh impact or future-month performance was established.

### Submission checks

- [x] The paper has an abstract at the top and all nine required sections, including FlyRank data credit at the bottom.
- [x] The analysis keeps label-window data, hashed identifiers, and product decisions out of model features.
- [x] Charts and receipts contain aggregate metrics only; no client identifiers, URLs, or private queries are exported.
- [ ] Run all cells in Colab with authorized `HF_TOKEN`, then save the completed notebook back to this repository.
- [ ] Confirm `submission/paper_url.txt` contains the deployed HTTPS URL on one line.

## Acknowledgments & data credit

Built on the [FlyRank ML Internship dataset](https://flyrank.ai/). The warehouse is gated and used under its data-use terms; all findings here are aggregate and decision-support only.